In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Configuration
base_dir = "./results/only_bomb_prompt"
confidence_level = 0.95  # 95% confidence interval

# List all model directories (format: provider/ModelName_layer or provider/ModelName_layer_sae)
model_dirs = [
    "meta-llama/Llama-3.1-8B-Instruct_11", 
    "meta-llama/Llama-3.1-8B-Instruct_16", 
    "meta-llama/Llama-3.1-8B-Instruct_21", 
    "meta-llama/Llama-3.1-8B-Instruct_19_sae",
    "meta-llama/Llama-3.1-8B-Instruct_19",
    "Qwen/Qwen2.5-7B-Instruct_14",
    "tiiuae/Falcon3-7B-Instruct_14",
]

In [ ]:
rows = []
for model_dir in model_dirs:
    print(model_dir)
    
    # Parse model information from new format: "provider/ModelName_layer" or "provider/ModelName_layer_sae"
    provider, model_part = model_dir.split('/')
    sae = model_part.endswith('_sae')
    
    if sae:
        # Remove _sae suffix and split
        model_part_clean = model_part.rsplit('_sae', 1)[0]
        parts = model_part_clean.rsplit('_', 1)
        model_name = parts[0]
        layer = int(parts[1])
    else:
        parts = model_part.rsplit('_', 1)
        model_name = parts[0]
        layer = int(parts[1])
    
    # Create a simple identifier for the model
    model_id = f"{provider}/{model_name}"
    
    eval_dir = os.path.join(base_dir, model_dir, "eval")
    if not os.path.exists(eval_dir):
        print(f"  Skipping - eval dir not found: {eval_dir}")
        continue
        
    # Get all JSON files
    json_files = [f for f in os.listdir(eval_dir) if f.endswith('.json')]
    
    # Extract coefficients from first file
    with open(os.path.join(eval_dir, json_files[0]), 'r') as f:
        data = json.load(f)
    coefs = sorted([float(k) for k in data["steered_responses"]["0"].keys()])
    
    # Create relative coefficients dynamically based on number of actual coefficients
    # Start from 0 for baseline (no steering), then 0.25, 0.5, etc. for actual steering
    num_coefs = len(coefs)
    relative_coefs = [0.25 * (i + 1) for i in range(num_coefs)]  # 0.25, 0.5, 0.75, 1.0, ...
    coef_mapping = dict(zip(sorted(coefs), relative_coefs))
    
    # Collect judgments for steered responses
    judgments = {coef: [] for coef in coefs}
    baseline_count = 0  # Count baseline samples (no steering = always safe)
    
    for json_file in json_files:
        with open(os.path.join(eval_dir, json_file), 'r') as f:
            data = json.load(f)
        # Count baseline (one per file)
        baseline_count += 1
        for feature_data in data["steered_responses"].values():
            for coef, response in feature_data.items():
                judgments[float(coef)].append(response["judgment"])
    
    # Add baseline (coef=0, no steering) - baseline is always safe (model refuses by default)
    # Baseline has 0% compliance rate (0 unsafe out of baseline_count samples)
    n_baseline = baseline_count
    k_baseline = 0  # All baselines are safe (no steering = model refuses)
    p_baseline = 0
    
    # Calculate 95% CI for baseline
    z_95 = stats.norm.ppf(1 - (1 - 0.95) / 2)
    denominator = 1 + z_95**2 / n_baseline
    centre = (p_baseline + z_95**2 / (2 * n_baseline)) / denominator
    half_width = z_95 * np.sqrt(p_baseline * (1 - p_baseline) / n_baseline + z_95**2 / (4 * n_baseline**2)) / denominator
    ci_low_baseline_95 = max(0, centre - half_width)
    ci_high_baseline_95 = min(1, centre + half_width)
    
    # Calculate 75% CI for baseline
    z_75 = stats.norm.ppf(1 - (1 - 0.75) / 2)
    denominator = 1 + z_75**2 / n_baseline
    centre = (p_baseline + z_75**2 / (2 * n_baseline)) / denominator
    half_width = z_75 * np.sqrt(p_baseline * (1 - p_baseline) / n_baseline + z_75**2 / (4 * n_baseline**2)) / denominator
    ci_low_baseline_75 = max(0, centre - half_width)
    ci_high_baseline_75 = min(1, centre + half_width)
    
    rows.append({
        "model_id": model_id,
        "model_name": model_name,
        "provider": provider,
        "layer": layer,
        "sae": sae,
        "coef": 0.0,
        "relative_coef": 0.0,  # 0 = no steering
        "unsuccess_prob": 100 * p_baseline,
        "ci_low": 100 * ci_low_baseline_95,
        "ci_high": 100 * ci_high_baseline_95,
        "ci_low_75": 100 * ci_low_baseline_75,
        "ci_high_75": 100 * ci_high_baseline_75,
        "n": n_baseline,
        'k': k_baseline,
    })
    
    # Calculate unsuccess probability with confidence intervals for steered responses
    for coef, judgment_list in judgments.items():
        n = len(judgment_list)
        k = sum(1 for j in judgment_list if j == "unsafe")
        p = k / n if n > 0 else 0
        
        # Calculate 95% CI using Wilson score interval
        if n > 0:
            z_95 = stats.norm.ppf(1 - (1 - 0.95) / 2)
            denominator = 1 + z_95**2 / n
            centre = (p + z_95**2 / (2 * n)) / denominator
            half_width = z_95 * np.sqrt(p * (1 - p) / n + z_95**2 / (4 * n**2)) / denominator
            ci_low_95 = max(0, centre - half_width)
            ci_high_95 = min(1, centre + half_width)
            
            # Calculate 75% CI
            z_75 = stats.norm.ppf(1 - (1 - 0.75) / 2)
            denominator = 1 + z_75**2 / n
            centre = (p + z_75**2 / (2 * n)) / denominator
            half_width = z_75 * np.sqrt(p * (1 - p) / n + z_75**2 / (4 * n**2)) / denominator
            ci_low_75 = max(0, centre - half_width)
            ci_high_75 = min(1, centre + half_width)
        else:
            ci_low_95, ci_high_95 = 0, 0
            ci_low_75, ci_high_75 = 0, 0
            
        rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "provider": provider,
            "layer": layer,
            "sae": sae,
            "coef": coef,
            "relative_coef": coef_mapping[coef],
            "unsuccess_prob": 100*p,
            "ci_low": 100*ci_low_95,
            "ci_high": 100*ci_high_95,
            "ci_low_75": 100*ci_low_75,
            "ci_high_75": 100*ci_high_75,
            "n": n,
            'k': k,
        })

df = pd.DataFrame(rows)
print(f"\nLoaded {len(df)} rows")
print(f"Models: {df['model_id'].unique()}")
print(f"Layers: {df['layer'].unique()}")
print(f"Relative coefs: {sorted(df['relative_coef'].unique())}")

In [ ]:
df[df['relative_coef'] == 0.25]

In [ ]:
def plot_llama_layers(y_max=None, log_scale=False, hide_y_axis=False):
    """Plot Llama 3.1 8B results across different layers."""
    fig, ax = plt.subplots(figsize=(10, 7.5))
    # Use fixed margins to keep plot area consistent regardless of y-axis labels
    plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.12)
    
    # Define colors and labels for each layer (using available layers: 11, 16, 21)
    layer_config = {
        11: {'color': "#45B7D1", 'label': 'Llama-3.1-8B layer 11 (1/3 depth, Random)'},
        16: {'color': "#e19327", 'label': 'Llama-3.1-8B layer 16 (1/2 depth, Random)'},
        21: {'color': "#4ECD96", 'label': 'Llama-3.1-8B layer 21 (2/3 depth, Random)'}
    }
    
    model_id = "meta-llama/Llama-3.1-8B-Instruct"
    
    for layer in [11, 16, 21]:
        subset = df[(df.model_id == model_id) & (df.layer == layer) & (df.sae == False)].sort_values('relative_coef')
        
        if len(subset) == 0:
            print(f"No data for layer {layer}")
            continue
        
        # Prepare error bars (use 75% CI for log scale, 95% for linear)
        ci_low_col = 'ci_low_75' if log_scale else 'ci_low'
        ci_high_col = 'ci_high_75' if log_scale else 'ci_high'
        y_err_lower = subset.unsuccess_prob - subset[ci_low_col]
        y_err_lower[y_err_lower < 0] = 0
        y_err_upper = subset[ci_high_col] - subset.unsuccess_prob
        # Set error bars to 0 when compliance rate is 0
        y_err_lower[subset.unsuccess_prob == 0] = 0
        y_err_upper[subset.unsuccess_prob == 0] = 0
        y_err = [y_err_lower.values, y_err_upper.values]

        plt.errorbar(
            subset.relative_coef, 
            subset.unsuccess_prob, 
            yerr=y_err,
            label=layer_config[layer]['label'],
            color=layer_config[layer]['color'],
            marker='o,
            capsize=12,
            linestyle='-',
            linewidth=10,
            markersize=15, 
            alpha=1,
        )

    plt.xlabel("Scaling Coefficient (Steering Strength)", fontsize=26)
    if not hide_y_axis:
        plt.ylabel("Compliance Rate (%)", fontsize=28)
    plt.grid(True, alpha=1)

    plt.xticks(fontsize=23)
    plt.yticks(fontsize=24)
    ax.legend(fontsize=18, loc='upper left')
    
    if log_scale:
        plt.yscale('symlog', linthresh=0.1)
        # Custom tick labels: show as "1%" instead of "10^0"
        ax = plt.gca()
        yticks = [0, 0.1, 1, 10, 100]
        ax.set_yticks(yticks)
        if hide_y_axis:
            ax.set_yticklabels([])
        else:
            ax.set_yticklabels(['0%', '0.1%', '1%', '10%', '100%'])
    elif hide_y_axis:
        plt.gca().set_yticklabels([])
    
    if y_max is not None:
        plt.ylim(0, y_max)

    plt.show()

# Example usage - linear scale
plot_llama_layers(y_max=25)
# Log scale
plot_llama_layers(y_max=100, log_scale=True)
# Log scale without y-axis (for combining)
plot_llama_layers(y_max=100, log_scale=True, hide_y_axis=True)

In [ ]:
def plot_sae_comparison(y_max=None, log_scale=False, hide_y_axis=False):
    """Compare Random steering vs SAE steering for Llama 3.1 8B."""
    fig, ax = plt.subplots(figsize=(10, 7.5))
    # Use fixed margins to keep plot area consistent regardless of y-axis labels
    plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.12)
    
    model_id = "meta-llama/Llama-3.1-8B-Instruct"
    
    # Define colors and markers for each SAE condition
    # Random is at layer 21, SAE is at layer 19
    sae_config = {
        (False, 19): {'color': '#e19327', 'marker': 's', 'label': 'Llama-3.1-8B layer 19 (Random)'},
        (True, 19): {'color': "#FF6B6B", 'marker': 's', 'label': 'Llama-3.1-8B layer 19 (SAE)'}
    }
    
    for (sae_flag, layer), config in sae_config.items():
        subset = df[(df.model_id == model_id) & (df.sae == sae_flag) & (df.layer == layer)].sort_values('relative_coef')
        
        if len(subset) == 0:
            print(f"No data for SAE={sae_flag}, layer={layer}")
            continue
        
        # Prepare error bars (use 75% CI for log scale, 95% for linear)
        ci_low_col = 'ci_low_75' if log_scale else 'ci_low'
        ci_high_col = 'ci_high_75' if log_scale else 'ci_high'
        y_err_lower = subset.unsuccess_prob - subset[ci_low_col]
        y_err_lower[y_err_lower < 0] = 0
        y_err_upper = subset[ci_high_col] - subset.unsuccess_prob
        # Set error bars to 0 when compliance rate is 0
        y_err_lower[subset.unsuccess_prob == 0] = 0
        y_err_upper[subset.unsuccess_prob == 0] = 0
        y_err = [y_err_lower.values, y_err_upper.values]

        plt.errorbar(
            subset.relative_coef, 
            subset.unsuccess_prob, 
            yerr=y_err,
            label=config['label'],
            color=config['color'],
            marker=config['marker'],
            capsize=10,
            linestyle='-',
            linewidth=10,
            markersize=15, 
            alpha=1,
        )

    plt.xlabel("Scaling Coefficient (Steering Strength)", fontsize=26)
    if not hide_y_axis:
        plt.ylabel("Compliance Rate (%)", fontsize=28)
    plt.grid(True, alpha=1)

    plt.xticks(fontsize=23)
    plt.yticks(fontsize=24)
    ax.legend(fontsize=20, loc='upper left')
    
    if log_scale:
        plt.yscale('symlog', linthresh=0.1)
        # Custom tick labels: show as "1%" instead of "10^0"
        ax = plt.gca()
        yticks = [0, 0.1, 1, 10, 100]
        ax.set_yticks(yticks)
        if hide_y_axis:
            ax.set_yticklabels([])
        else:
            ax.set_yticklabels(['0%', '0.1%', '1%', '10%', '100%'])
    elif hide_y_axis:
        plt.gca().set_yticklabels([])
    
    if y_max is not None:
        plt.ylim(0, y_max)
    
    plt.show()

# Example usage - linear scale
plot_sae_comparison(y_max=25)
# Log scale
plot_sae_comparison(y_max=100, log_scale=True)
# Log scale without y-axis (for combining)
plot_sae_comparison(y_max=100, log_scale=True, hide_y_axis=True)

In [ ]:
def plot_model_comparison(y_max=None, log_scale=False, hide_y_axis=False):
    """Compare different models (Llama, Qwen, Falcon) at similar relative depths."""
    fig, ax = plt.subplots(figsize=(10, 7.5))
    # Use fixed margins to keep plot area consistent regardless of y-axis labels
    plt.subplots_adjust(left=0.12, right=0.95, top=0.95, bottom=0.12)
    
    # Define configuration for each model with unique colors, markers, and labels
    # Using available models and their layers
    model_config = {
        ("meta-llama/Llama-3.1-8B-Instruct", 16): {
            'color': '#45B7D1', 
            'marker': 's', 
            'label': 'Llama-3.1-8B layer 16 (1/2 depth, Random)'
        },
        ("Qwen/Qwen2.5-7B-Instruct", 14): {
            'color': "#4ECD96", 
            'marker': 's', 
            'label': 'Qwen2.5-7B layer 14 (1/2 depth, Random)'
        },
        ("tiiuae/Falcon3-7B-Instruct", 14): {
            'color': '#e19327', 
            'marker': 's', 
            'label': 'Falcon3-7B layer 14 (1/2 depth, Random)'
        }
    }
    
    for (model_id, layer), config in model_config.items():
        subset = df[(df.model_id == model_id) & (df.layer == layer) & (df.sae == False)].sort_values('relative_coef')
        
        if len(subset) == 0:
            print(f"No data for {model_id} layer {layer}")
            continue
        
        # Prepare error bars (use 75% CI for log scale, 95% for linear)
        ci_low_col = 'ci_low_75' if log_scale else 'ci_low'
        ci_high_col = 'ci_high_75' if log_scale else 'ci_high'
        y_err_lower = subset.unsuccess_prob - subset[ci_low_col]
        y_err_lower[y_err_lower < 0] = 0
        y_err_upper = subset[ci_high_col] - subset.unsuccess_prob
        # Set error bars to 0 when compliance rate is 0
        y_err_lower[subset.unsuccess_prob == 0] = 0
        y_err_upper[subset.unsuccess_prob == 0] = 0
        y_err = [y_err_lower.values, y_err_upper.values]
        
        plt.errorbar(
            subset.relative_coef, 
            subset.unsuccess_prob, 
            yerr=y_err,
            label=config['label'],
            color=config['color'],
            marker=config['marker'],
            capsize=10,
            linestyle='-',
            linewidth=10,
            markersize=15, 
            alpha=1,
        )

    plt.xlabel("Scaling Coefficient (Steering Strength)", fontsize=26)
    if not hide_y_axis:
        plt.ylabel("Compliance Rate (%)", fontsize=28)
    plt.grid(True, alpha=1)

    plt.xticks(fontsize=23)
    plt.yticks(fontsize=24)
    ax.legend(fontsize=19, loc='upper left')
    
    if log_scale:
        plt.yscale('symlog', linthresh=0.1)
        # Custom tick labels: show as "1%" instead of "10^0"
        ax = plt.gca()
        yticks = [0, 0.1, 1, 10, 100]
        ax.set_yticks(yticks)
        if hide_y_axis:
            ax.set_yticklabels([])
        else:
            ax.set_yticklabels(['0%', '0.1%', '1%', '10%', '100%'])
    elif hide_y_axis:
        plt.gca().set_yticklabels([])
    
    if y_max is not None:
        plt.ylim(0, y_max)
    
    plt.show()

# Example usage - linear scale
plot_model_comparison(y_max=25)
# Log scale
plot_model_comparison(y_max=100, log_scale=True)
# Log scale without y-axis (for combining)
plot_model_comparison(y_max=100, log_scale=True, hide_y_axis=True)

In [ ]:
# ============================================
# Combined plot: All 3 log scale figures side by side
# ============================================

def plot_combined_log_scale():
    """Plot all 3 figures side by side with shared Y axis label on left only."""
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    
    model_id = "meta-llama/Llama-3.1-8B-Instruct"

    # ---- Subplot 1: Model Comparison ----
    ax = axes[0]
    model_config = {
        ("meta-llama/Llama-3.1-8B-Instruct", 16): {'color': '#e19327', 'label': 'Llama-3.1-8B Layer 16 (1/2 depth, Random)', 'marker': 's', 'linewidth': 12},
        ("Qwen/Qwen2.5-7B-Instruct", 14): {'color': "#4ECD96", 'label': 'Qwen2.5-7B Layer 14 (1/2 depth, Random)', 'marker': 's', 'linewidth': 6},
        ("tiiuae/Falcon3-7B-Instruct", 14): {'color': '#45B7D1', 'label': 'Falcon3-7B Layer 14 (1/2 depth, Random)', 'marker': 's', 'linewidth': 10}
    }
    
    for (mid, layer), config in model_config.items():
        subset = df[(df.model_id == mid) & (df.layer == layer) & (df.sae == False)].sort_values('relative_coef')
        if len(subset) == 0:
            continue
        
        y_err_lower = subset.unsuccess_prob - subset['ci_low_75']
        y_err_lower[y_err_lower < 0] = 0
        y_err_upper = subset['ci_high_75'] - subset.unsuccess_prob
        y_err_lower[subset.unsuccess_prob == 0] = 0
        y_err_upper[subset.unsuccess_prob == 0] = 0
        y_err = [y_err_lower.values, y_err_upper.values]

        ax.errorbar(
            subset.relative_coef, subset.unsuccess_prob, yerr=y_err,
            label=config['label'], color=config['color'],
            marker=config.get('marker', 'o'),
            capsize=10, linestyle='-', linewidth=config['linewidth'], markersize=20, alpha=1,
            ecolor='black', elinewidth=2, capthick=2,
        )
    
    ax.set_ylabel("Compliance Rate (%)", fontsize=30, labelpad=5)
    ax.grid(True, alpha=1)
    ax.legend(fontsize=16, loc='upper left')
    ax.set_yscale('symlog', linthresh=0.1)
    ax.set_yticks([0, 0.1, 1, 10, 100])
    ax.set_yticklabels(['0%', '0.1%', '1%', '10%', '100%'])
    ax.tick_params(axis='both', labelsize=22)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelleft=True)  # Ensure y-tick labels are shown
    ax.set_ylim(0, 100)

    # ---- Subplot 2: Llama Layers ----
    ax = axes[1]
    layer_config = {
        11: {'color': "#45B7D1", 'label': 'Llama-3.1-8B Layer 11 (1/3 depth, Random)', 'marker': 's'},
        16: {'color': "#e19327", 'label': 'Llama-3.1-8B Layer 16 (1/2 depth, Random)', 'marker': 's'},
        21: {'color': "#4ECD96", 'label': 'Llama-3.1-8B Layer 21 (2/3 depth, Random)', 'marker': 's'}
    }
    
    for layer in [11, 16, 21]:
        subset = df[(df.model_id == model_id) & (df.layer == layer) & (df.sae == False)].sort_values('relative_coef')
        if len(subset) == 0:
            continue
        
        y_err_lower = subset.unsuccess_prob - subset['ci_low_75']
        y_err_lower[y_err_lower < 0] = 0
        y_err_upper = subset['ci_high_75'] - subset.unsuccess_prob
        y_err_lower[subset.unsuccess_prob == 0] = 0
        y_err_upper[subset.unsuccess_prob == 0] = 0
        y_err = [y_err_lower.values, y_err_upper.values]

        ax.errorbar(
            subset.relative_coef, subset.unsuccess_prob, yerr=y_err,
            label=layer_config[layer]['label'], color=layer_config[layer]['color'],
            marker=layer_config[layer].get('marker', 'o'),
            capsize=10, linestyle='-', linewidth=10, markersize=20, alpha=1,
            ecolor='black', elinewidth=2, capthick=2,
        )

    ax.grid(True, alpha=1)
    ax.legend(fontsize=15, loc='upper left')
    ax.set_yscale('symlog', linthresh=0.1)
    ax.set_yticks([0, 0.1, 1, 10, 100])
    ax.set_yticklabels([])  # No y-tick labels for middle plot
    ax.tick_params(axis='x', labelsize=18)
    ax.set_ylim(0, 100)
    
    # ---- Subplot 3: SAE Comparison ----
    ax = axes[2]
    sae_config = {
        (False, 19): {'color': '#e19327', 'label': 'Llama3.1-8B Layer 19 (Random)', 'marker': 's'},
        (True, 19): {'color': "#FF6B6B", 'label': 'Llama3.1-8B Layer 19 (SAE)', 'marker': 's'}
    }
    
    for (sae_flag, layer), config in sae_config.items():
        subset = df[(df.model_id == model_id) & (df.sae == sae_flag) & (df.layer == layer)].sort_values('relative_coef')
        if len(subset) == 0:
            continue
        
        y_err_lower = subset.unsuccess_prob - subset['ci_low_75']
        y_err_lower[y_err_lower < 0] = 0
        y_err_upper = subset['ci_high_75'] - subset.unsuccess_prob
        y_err_lower[subset.unsuccess_prob == 0] = 0
        y_err_upper[subset.unsuccess_prob == 0] = 0
        y_err = [y_err_lower.values, y_err_upper.values]

        ax.errorbar(
            subset.relative_coef, subset.unsuccess_prob, yerr=y_err,
            label=config['label'], color=config['color'],
            marker=config.get('marker', 'o'),
            capsize=10, linestyle='-', linewidth=10, markersize=22, alpha=1,
            ecolor='black', elinewidth=2, capthick=2,
        )
    
    ax.grid(True, alpha=1)
    ax.legend(fontsize=20, loc='upper left')
    ax.set_yscale('symlog', linthresh=0.1)
    ax.set_yticks([0, 0.1, 1, 10, 100])
    ax.set_yticklabels([])  # No y-tick labels for right plot
    ax.tick_params(axis='x', labelsize=18)
    ax.set_ylim(0, 100)
    
    # Add shared x label for all figures
    fig.text(0.537, 0.00, "Scaling Coefficient (Steering Strength)", ha='center', fontsize=26)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15, left=0.08, wspace=0.03)  # tighter inter-panel spacing
    plt.show()

plot_combined_log_scale()
